- 해설서를 요약본.json / 전체.json 두 가지로 생성
- parent_id 를 붙여 ParentDocumentRetriver로 묶어준다. 

In [10]:
import pandas as pd 

sample_33 = pd.read_csv('./datasets/samples_33.csv')
sample_33.head(1)
sample_33['hs_description'][10]

"'관세율표 제3301호에 따르면 정유는 향료ㆍ식품 및 기타의 공업에서 원료로 사용되며 그 원천이 식물인 제반 제품 중 하나로 분류됩니다. 대부분의 정유는 휘발성이며, 종이 위에 묻으면 곧 사라지는 특징을 갖고 있습니다.'  \n'해설서 (A)에는 수증기증류법을 구체적으로 예시로 들어 설명하고 있으며, 이에 따라, 본 품목은 수증기 증류한 오레가노 정유로 해석되어 관세율표의 해석에 따라 제3301299000호에 분류됩니다.'  \n'따라서, 관세율표의 해석에 관한 통칙 제1호 및 제6호에 따라 제3301299000호에 분류함을 명시합니다. 해당 제품은 정유로서 적합한 관세율표에 의거하여 분류됩니다.'"

# JSON 변환 (해설서)

In [2]:
import pandas as pd
import json
import math

# 파일 경로 (업로드된 해설서 CSV 기준)
guide_path = "./datasets/hs_33.csv"
guide_df = pd.read_csv(guide_path)

summary_json = []
chunk_json = []

for idx, row in guide_df.iterrows():
    hs_code = str(row['HS_CD'])
    kor_name = row['HS_KOR_NAME']
    eng_name = row['HS_ENG_NAME']
    mti_code = str(row['MTI_CD']) if not pd.isna(row['MTI_CD']) else "정보 없음"
    mti_name = row['MTI_KOR_NAME'] if not pd.isna(row['MTI_KOR_NAME']) else "정보 없음"

    # 수출입 및 건수 숫자 처리
    def safe_num(val):
        return "정보 없음" if pd.isna(val) or not isinstance(val, (int, float)) or math.isnan(val) else f"{int(val):,}"

    exp_amt = safe_num(row['EXP_AMT'])
    imp_amt = safe_num(row['IMP_AMT'])
    count = safe_num(row['count'])

    # 법령 해설
    ryu_ex = row['ryu_ex'] if isinstance(row['ryu_ex'], str) else "정보 없음"
    ho_ex = row['ho_ex'] if isinstance(row['ho_ex'], str) else "정보 없음"

    # 요약 content 생성
    summary_content = (
        f"HS CODE {hs_code}는 '{kor_name}({eng_name})'에 해당하는 품목입니다. "
        f"MTI 코드 {mti_code}({mti_name})로 분류되며, 최근 통계에 따르면 수출금액은 {exp_amt} 달러, "
        f"수입금액은 {imp_amt} 달러, 신고 건수는 {count}건입니다. 이 품목은 제33류에 속하며, 식물에서 추출한 정유 또는 방향성 물질일 수 있습니다."
    )

    # 메타데이터 공통
    metadata = {
        "hs_code": hs_code,
        "product_name": kor_name,
        "mti_code": mti_code,
        "document_type": "guide",
        "parent_id": f"HS_{hs_code}"
    }

    # summary 문서
    summary_json.append({
        "id": f"HS_{hs_code}_summary",
        "content": summary_content,
        "metadata": metadata
    })

    # chunk 문서 (류 해설, 호 해설 각각)
    if ryu_ex != "정보 없음":
        chunk_json.append({
            "id": f"HS_{hs_code}_ryu",
            "content": f"[류 해설]\n{ryu_ex.strip()}",
            "metadata": metadata
        })

    if ho_ex != "정보 없음":
        chunk_json.append({
            "id": f"HS_{hs_code}_ho",
            "content": f"[호 해설]\n{ho_ex.strip()}",
            "metadata": metadata
        })

# JSON 저장
with open("./datasets/hs_33_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, ensure_ascii=False, indent=2)

with open("./datasets/hs_33_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunk_json, f, ensure_ascii=False, indent=2)


# JSON 변환 (분류 사례)

In [3]:
import pandas as pd
import json
import math

# CSV 로드 (사례 데이터)
case_path = "./datasets/samples_33.csv"
case_df = pd.read_csv(case_path)

# 결과 저장 리스트
case_summary_json = []
case_chunks_json = []

for idx, row in case_df.iterrows():
    hs_code = str(row["HS_CODE"])
    product = row["product"]
    description = row["pd_description"]
    reasoning = row["hs_description"]
    agency = row["agency"]
    date = str(row["date"])

    # 간단 요약 생성 (summary.json 용)
    summary_content = (
        f"'{product}'는 HS CODE {hs_code}로 분류된 사례입니다. "
        f"이 품목은 {agency}에 의해 {date}에 분류 결정이 내려졌으며, 향료 또는 화장품 관련 제품으로 보입니다."
    )

    summary_metadata = {
        "hs_code": hs_code,
        "agency": agency,
        "date": date,
        "document_type": "case_summary",
        "parent_id": f"HS_{hs_code}"
    }

    case_summary_json.append({
        "id": f"case_{hs_code}_{idx}_summary",
        "content": summary_content,
        "metadata": summary_metadata
    })

    # chunk 1 - 제품 설명
    if isinstance(description, str) and len(description.strip()) > 0:
        case_chunks_json.append({
            "id": f"case_{hs_code}_{idx}_desc",
            "content": f"[제품 설명]\n{description.strip()}",
            "metadata": {
                **summary_metadata,
                "document_type": "case_chunk"
            }
        })

    # chunk 2 - 분류 사유
    if isinstance(reasoning, str) and len(reasoning.strip()) > 0:
        case_chunks_json.append({
            "id": f"case_{hs_code}_{idx}_reason",
            "content": f"[분류 사유]\n{reasoning.strip()}",
            "metadata": {
                **summary_metadata,
                "document_type": "case_chunk"
            }
        })

# 저장
with open("./datasets/samples_33_summary.json", "w", encoding="utf-8") as f:
    json.dump(case_summary_json, f, ensure_ascii=False, indent=2)

with open("./datasets/samples_33_chunks.json", "w", encoding="utf-8") as f:
    json.dump(case_chunks_json, f, ensure_ascii=False, indent=2) 


# ParentDocumentRetriver 예시 

In [ ]:
from langchain.vectorstores import Chroma
from langchain.storage import InMemoryStore
from langchain.retrievers import ParentDocumentRetriever
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings

# 임베딩 모델 설정
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# summary + chunks 로드
from langchain.document_loaders import JSONLoader
summary_docs = JSONLoader(file_path="guide_summary.json", jq_schema=".[]", text_content=False).load()
chunk_docs = JSONLoader(file_path="guide_chunks.json", jq_schema=".[]", text_content=False).load()

# child용 vectorstore와 docstore 준비
# 전체 해설서 JSON이 
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
vectorstore = Chroma.from_documents(chunk_docs, embedding, collection_name="hs_chunks")
docstore = InMemoryStore()

# ParentDocumentRetriever 생성
retriever = ParentDocumentRetriever.from_components(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_documents=summary_docs
)
